# Libraries

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import json
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import random
import optuna
from pathlib import Path

# -- Personal Libraries
from src.dominick import DominickDataLoader
from src.dominick.multiproduct_builder import MultiProductBuilder
from src.nn.data import ColumnEncoder, DataLoaderFactory, SplineBuilder
from src.nn.spline import build_price_basis
from src.nn.models import IntegrableDemandHead, ICDN
from src.nn.loss import ElasticityLoss
from src.multiproduct import MultiProductDataset, ProductTokenBuilder
from src.utils import TemporalSplitter

/home/thebigmonster/Github/nn-elasticity/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Setting

In [ ]:
# initial seed
BASE_SEED = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# ── Data ──────────────────────────────────────────────────────────
N_UPCS = 5 # Number of UPCs
SMOOTH_WINDOW = 8 # Smoothing window for phase 0
BETA_EDA = -2 # Beta for initialization phase 0
K_NEIGHBORS = 5 # Number of neighbors for the product

# ── Nested temporal tuning (budget-matched vs 101 leaky trials) ──
PROTOCOL = "nested_temporal"
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 3
TUNE_SEEDS = [11, 29, 42]
MIN_TRAIN_FRAC = 0.50
TRAIN_FRAC = 0.8
N_TRIALS_NESTED = 20
N_TRIALS_HOLDOUT = 20

# ── Training for tuning ──────────────────────────────────────
N_EPOCHS_P0 = 250
N_EPOCHS_P1 = 300
PATIENCE    = 30 # How many epochs to wait before reducing learning rate
ES_PATIENCE = 70 # How many epochs to wait before early stopping

# ── Dimensionality for sku-level features ───────────────────────────
D_STORE = 16
D_BRAND = 8
D_STYLE = 8

# ── Checkpoints ────────────────────────────────────────────────────
CKPT_DIR = Path("../results/checkpoints/hparam_nested")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Results ─────────────────────────────────────────────────────
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
NESTED_DIR = RESULTS_DIR / "nested"
NESTED_DIR.mkdir(parents=True, exist_ok=True)

Device: cuda


# Seeds

In [4]:
# Function to set all seeds
# and make the results reproducible
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True # Make the results reproducible and control the randomness
    torch.backends.cudnn.benchmark = False # Make the results reproducible and control the randomness

set_all_seeds(BASE_SEED)

# Loader

In [ ]:
# Load the dataset
loader = DominickDataLoader()
df = loader.load("elasticity_dataset.csv").copy()
print(f"Dataset shape: {df.shape}")

# Encode the categorical variables
encoder = ColumnEncoder()
_, store_cats = encoder.factorize(df, "store_code", sort=True) # Encode the store code to numerical values
_, week_cats  = encoder.factorize(df, "week_id", sort=True) # Encode the week id to numerical values
_, brand_cats = encoder.factorize(df, "brand_family_norm",  sort=True)   # Encode the brand family to numerical values
_, style_cats = encoder.factorize(df, "style_segment_norm", sort=True)   # Encode the style segment to numerical values

n_stores = len(store_cats)
n_weeks  = len(week_cats)
n_brands = len(brand_cats)
n_styles = len(style_cats)
print(f"Stores: {n_stores}  |  Weeks: {n_weeks}  |  Brands: {n_brands}  |  Styles: {n_styles}")

# Encode brand y style en el dataframe principal
# We create a mapping of brand and style (numerical) codes to 0,1,2,...
# to be globally used for the folds; For instance,
# brand_cats = Index([101, 102,...])
# brand_map = {101: 0, 102: 1, ...}
# The same for style_cats and style_map.          
brand_map = {v: i + 1 for i, v in enumerate(brand_cats)}
style_map = {v: i + 1 for i, v in enumerate(style_cats)}
df["brand_family_norm"]  = df["brand_family_norm"].map(brand_map).fillna(0).astype(int)
df["style_segment_norm"] = df["style_segment_norm"].map(style_map).fillna(0).astype(int)

# Build the multi-product dataset
mp_builder = MultiProductBuilder()

sorted_weeks = sorted(df["week_id"].unique())
n_sample = max(1, int(len(sorted_weeks) * MIN_TRAIN_FRAC))
sample_weeks = sorted_weeks[:n_sample]
mp_builder.fit_panel(df[df["week_id"].isin(sample_weeks)], n_upcs=N_UPCS)
n_upcs = mp_builder.n
upc_names = mp_builder.selected_upcs

print(f"Selection window: {n_sample} weeks (never used as outer val)")
print(f"UPCs selected: {list(upc_names)}")
print(f"Top UPCs: {mp_builder.selected_upcs[:N_UPCS]}")

Dataset shape: (463722, 44)
Stores: 70  |  Weeks: 302  |  Brands: 54  |  Styles: 13
Full wide shape: (19808, 171)
UPCs selected: 5
Top UPCs: [3410010505, 7289000011, 1820000784, 8248812345, 3410017306]


# Neighbor Meta

In [6]:
# Neighbors:
# Static metadata per UPC position — used by neighbor-aware attention in the model
upc_meta = (
    df.groupby("upc_code")[["category_code", "brand_family_norm",
                             "style_segment_norm", "liters_per_upc"]]
    .first()
    .loc[mp_builder.selected_upcs]
)

cat_codes, _ = pd.factorize(upc_meta["category_code"], sort=True)

neighbor_meta = {
    "category": torch.tensor(cat_codes, dtype=torch.long, device=device),
    "brand":    torch.tensor(upc_meta["brand_family_norm"].values,   dtype=torch.long,    device=device),
    "style":    torch.tensor(upc_meta["style_segment_norm"].values,  dtype=torch.long,    device=device),
    "liters":   torch.tensor(upc_meta["liters_per_upc"].values,      dtype=torch.float32, device=device),
}
print("neighbor_meta built")

neighbor_meta built


# Temporal Folds

In [ ]:
splitter = TemporalSplitter(week_col="week_id")

nested_plans = splitter.nested_expanding_splits(
    df=df,
    n_outer=N_OUTER_FOLDS,
    n_inner=N_INNER_FOLDS,
    min_train_frac=MIN_TRAIN_FRAC,
)

def materialize(plan_or_pair):
    if isinstance(plan_or_pair, dict):
        tr_w, va_w = mp_builder.make_fold_frames(plan_or_pair["outer_train"], plan_or_pair["outer_val"])
        plan_or_pair["outer_train"], plan_or_pair["outer_val"] = tr_w, va_w
        plan_or_pair["inner_splits"] = [
            mp_builder.make_fold_frames(tr, va) for tr, va in plan_or_pair["inner_splits"]
        ]
        return plan_or_pair
    tr, va = plan_or_pair
    return mp_builder.make_fold_frames(tr, va)

nested_plans = [materialize(p) for p in nested_plans]

train_final_long, val_final_long = splitter.single_split(df, train_frac=TRAIN_FRAC)
holdout_inner_long = splitter.expanding_splits(
    train_final_long, n_folds=N_INNER_FOLDS, min_train_frac=MIN_TRAIN_FRAC,
)
holdout_weeks = set(val_final_long["week_id"].unique())
for _, inner_val in holdout_inner_long:
    leak = set(inner_val["week_id"].unique()) & holdout_weeks
    if leak:
        raise RuntimeError(f"Holdout leak: {sorted(leak)[:10]}")

holdout_inner = [mp_builder.make_fold_frames(tr, va) for tr, va in holdout_inner_long]
train_final, val_final = mp_builder.make_fold_frames(train_final_long, val_final_long)
train_weeks_final = sorted(train_final["week_id"].unique())

# Functions

In [ ]:
# We create a mapping of store and week (numerical)codes to 0,1,2,...
# to be globally used for the folds; For instance,
# store_cats = Index([101, 102,...])
# store_map = {101: 0, 102: 1, ...}
# The same for week_cats and week_map.
store_map = {v: i for i, v in enumerate(store_cats)}
week_map  = {v: i for i, v in enumerate(week_cats)}


# This function prepare the data for training.
# It encodes the store and week codes, sorts the data by store and week codes,
# and smooths the log liters.
def build_fold_frames(train_wide, val_wide, smooth_window: int):
    train_wide = train_wide.copy()
    val_wide   = val_wide.copy()

    for w in [train_wide, val_wide]:
        w["store_code"] = w["store_code"].map(store_map)
        w["week_id"]    = w["week_id"].map(week_map)

    def _smooth(df_w):
        out = df_w.sort_values(["store_code", "week_id"]).copy()
        for i in range(n_upcs):
            y = out[f"log_liters_{i}"].where(out[f"obs_mask_{i}"].eq(1))
            out[f"log_liters_{i}"] = (
                y.groupby(out["store_code"])
                 .transform(lambda s: s.rolling(window=smooth_window, min_periods=1).mean())
                 .fillna(0.0)
            )
        return out

    train_wide_s = _smooth(train_wide)
    val_wide_s   = _smooth(val_wide)
    return train_wide, val_wide, train_wide_s, val_wide_s

# This function builds the datasets for the training and validation.
def build_loaders(train_wide, val_wide, train_wide_s, val_wide_s, batch_size: int):

    loader_factory = DataLoaderFactory(
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
    )

    # Create the DataLoaders for the phase0 and phase1.
    # For training we shuffle the data and drop the last batch.
    # For validation we don't shuffle the data and don't drop the last batch.
    # Important! One might think that shuffling the data could alter its sequential order,
    # however, in this case, the MLP will process the data for each pair (shop, week)
    # and, therefore, the order does not matter. It would be a problem if the architecture were, for example,
    # an RNN or an LSTM, but in this case it is not.
    # Observation! The drop_last is True for the training set. We try to avoid things like: 
    # 28 observations in the last batch compared to 500 in the others, for instance.

    train_ds_p0 = MultiProductDataset(train_wide_s, n=n_upcs) # Phase 0 training dataset
    val_ds_p0   = MultiProductDataset(val_wide_s,   n=n_upcs) # Phase 0 validation dataset
    train_ds    = MultiProductDataset(train_wide,   n=n_upcs) # Phase 1 training dataset
    val_ds      = MultiProductDataset(val_wide,     n=n_upcs) # Phase 1 validation dataset

    train_loader_p0 = loader_factory.create_train_loader(train_ds_p0, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader_p0   = loader_factory.create_eval_loader(val_ds_p0,   batch_size=batch_size, shuffle=False)
    train_loader    = loader_factory.create_train_loader(train_ds,    batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader      = loader_factory.create_eval_loader(val_ds,       batch_size=batch_size, shuffle=False)
    return train_loader_p0, val_loader_p0, train_loader, val_loader

# Helpers

In [ ]:
def zero_and_freeze_nonlinear(model):
    # Zero and freeze spline heads (own and cross) and the bilinear head.
    # With W = b = 0, w(h) = w_cross(h) = U(h) = 0 for any h, so Phase 0 is
    # exactly log-linear:
    #   g_i ≈ b_i + β_{ii}·u_i + Σ_j a_{ij}·β_{ij}·u_j
    # Only head_b, head_beta, head_beta_cross remain trainable.
    ph = model.head.param_head
    for attr in ("head_w", "head_w_cross", "head_cross"):
        if not hasattr(ph, attr):
            continue
        layer = getattr(ph, attr)
        with torch.no_grad():
            layer.weight.zero_()
            if layer.bias is not None:
                layer.bias.zero_()
        layer.weight.requires_grad_(False)
        if layer.bias is not None:
            layer.bias.requires_grad_(False)

def unfreeze_nonlinear(model):
    # Unfreeze all spline and bilinear heads for phase 1.
    for attr in ["head_w", "head_w_cross", "head_cross"]:
        head = getattr(model.head.param_head, attr)
        head.weight.requires_grad_(True)
        head.bias.requires_grad_(True)

# To initialize the beta prior of the model
# because of EDA, the global elasticity is -2.
def init_beta_prior(model, beta_target):
    beta_raw_init = torch.log(
        torch.exp(torch.tensor(-beta_target, dtype=torch.float32)) - 1.0
    )# Initialize the head_beta bias with the inverse softplus of BETA_EDA
    with torch.no_grad():
        # Set the head_beta weight to zero, therefore, the initial head_beta 
        # is independent of the context.
        model.head.param_head.head_beta.weight.zero_()
        # Set the head_beta bias with the inverse softplus of BETA_EDA.
        model.head.param_head.head_beta.bias.fill_(beta_raw_init)
        # This implies that beta_raw = 0*h + beta_raw_init = beta_raw_init
        # All products have the same beta_raw_init at the beginning. When
        # the model is trained, beta_raw will be updated.

def active_cross_mask(E, obs_mask, pairs):
    """True only on observed, selected directed edges (not the diagonal)."""
    n = E.shape[1]
    active = torch.zeros(n, n, dtype=torch.bool, device=E.device)
    if pairs is not None and pairs.numel() > 0:
        active[pairs[0], pairs[1]] = True
    obs = obs_mask.bool()
    return obs.unsqueeze(2) & obs.unsqueeze(1) & active.unsqueeze(0)

def _empty_loss_acc():
    return dict(fit_num=0.0, fit_den=0.0, sm_num=0.0, sm_den=0.0, el_num=0.0, el_den=0.0)

def _accum_loss(acc, logs):
    n_fit = logs["n_obs"].item()
    n_sm  = logs["n_smooth"].item()
    n_el  = logs["n_elast"].item()
    acc["fit_num"] += logs["loss_fit"].item()    * n_fit
    acc["fit_den"] += n_fit
    acc["sm_num"]  += logs["loss_smooth"].item() * n_sm
    acc["sm_den"]  += n_sm
    acc["el_num"]  += logs["loss_elast"].item()  * n_el
    acc["el_den"]  += n_el

def _epoch_loss(acc, loss_fn):
    return (
        acc["fit_num"] / max(acc["fit_den"], 1.0)
        + loss_fn.lambda_smooth * (acc["sm_num"] / max(acc["sm_den"], 1.0))
        + loss_fn.lambda_elast  * (acc["el_num"] / max(acc["el_den"], 1.0))
    )

print("Helpers defined")

Helpers defined


In [ ]:
def run_training(model, train_loader, val_loader, loss_fn,
                 optimizer, scheduler, n_epochs, es_patience,
                 ckpt_path, device, neighbor_meta, phase_name="",
                 verbose=False):
    """val_loader is None → fixed-length train, last-epoch checkpoint, no ES."""
    use_val = val_loader is not None
    best_val_loss = float("inf")
    best_epoch = n_epochs
    no_improve = 0
    scaler = torch.amp.GradScaler("cuda") if device == "cuda" else None

    for epoch in range(n_epochs):
        # ── Train ──────────────────────────────────────────────────
        model.train()
        train_acc = _empty_loss_acc()

        for batch in train_loader:
            batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            y_true   = batch["demands"]
            obs_mask = batch["obs_mask"]

            optimizer.zero_grad()
            if scaler:
                with torch.amp.autocast("cuda"):
                    y_hat, _, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
                    loss, logs = loss_fn(y_hat, y_true, obs_mask,
                                        aux["w"], aux["ddBx"], aux["u"], aux["Bx"],
                                        aux["pairs"], E=aux.get("E"), attn_weights=aux.get("attn_weights"),
                                        availability=batch["availability"], price_observed=batch["price_observed"])
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                y_hat, _, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
                loss, logs = loss_fn(y_hat, y_true, obs_mask,
                                    aux["w"], aux["ddBx"], aux["u"], aux["Bx"],
                                    aux["pairs"], E=aux.get("E"), attn_weights=aux.get("attn_weights"),
                                    availability=batch["availability"], price_observed=batch["price_observed"])
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            _accum_loss(train_acc, logs)

        if not use_val:
            scheduler.step()
            continue

        # ── Val ────────────────────────────────────────────────────
        model.eval()
        val_acc = _empty_loss_acc()

        with torch.no_grad():
            for batch in val_loader:
                batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
                y_true   = batch["demands"]
                obs_mask = batch["obs_mask"]

                y_hat, _, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
                _, logs = loss_fn(y_hat, y_true, obs_mask,
                                aux["w"], aux["ddBx"], aux["u"], aux["Bx"],
                                aux["pairs"], E=aux.get("E"), attn_weights=aux.get("attn_weights"),
                                price_observed=batch["price_observed"], availability=batch["availability"])

                _accum_loss(val_acc, logs)

        val_loss = _epoch_loss(val_acc, loss_fn)
        prev_lr = optimizer.param_groups[0]["lr"]
        scheduler.step(val_loss)
        new_lr = optimizer.param_groups[0]["lr"]
        if new_lr < prev_lr:
            no_improve = 0

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            no_improve = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            no_improve += 1

        if verbose and ((epoch + 1) % 50 == 0 or no_improve == 0):
            print(f"  [{phase_name}] Epoch {epoch+1}  val={val_loss:.4f}")

        if no_improve >= es_patience:
            if verbose:
                print(f"  [{phase_name}] Early stopping in epoch {epoch+1}")
            break

    if not use_val:
        torch.save(model.state_dict(), ckpt_path)

    return {"best_epoch": best_epoch, "best_val_loss": best_val_loss}

In [ ]:
# Hidden options for the model (Optuna)
HIDDEN_OPTIONS = {
    "64_32":        (64, 32),
    "128_64":       (128, 64),
    "192_96":       (192, 96),
    "256_128":      (256, 128),
    "256_128_64":   (256, 128, 64),
}

# This function compute the R2, MAE and RMSE.
# Recall that:
# MAE is the mean absolute error.
# RMSE is the root mean square error.
# R2 is the coefficient of determination.
def compute_global_metrics(model, val_loader, device):
    model.eval()
    all_true, all_pred = [], []

    with torch.no_grad():
        for batch in val_loader:
            batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            # demands and obs_mask are already (B, n) — pre-stacked in MultiProductDataset.__init__.
            y_true   = batch["demands"]   # (B, n) float — no torch.stack() needed
            obs_mask = batch["obs_mask"]  # (B, n) float — no torch.stack() needed
            y_hat, _, _ = model(batch, return_parts=True, neighbor_meta=neighbor_meta)

            # We get only the available observations.
            mask = obs_mask.bool() # Mask of the available observations
            all_true.append(y_true[mask].cpu()) # Append the true values
            all_pred.append(y_hat[mask].cpu()) # Append the predicted values

    y_true_all = torch.cat(all_true).float() # Concatenate the true values
    y_pred_all = torch.cat(all_pred).float() # Concatenate the predicted values

    err = y_true_all - y_pred_all # Error
    mae = float(err.abs().mean()) # Mean absolute error
    rmse = float(torch.sqrt((err ** 2).mean())) # Root mean square error

    ss_res = float((err ** 2).sum()) # Sum of the squared errors
    ss_tot = float(((y_true_all - y_true_all.mean()) ** 2).sum()) # Sum of the total errors
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan # R2

    return {
        "mae_val": mae,
        "rmse_val": rmse,
        "r2_val": r2,
    }

# This function compute the Elasticity Score for the optimization parameters of Optuna.
# Our intention is to evaluate how good the model is at predicting the elasticity.
# Recall that:
# The Elasticity Score is in the range [0, 1].
# The closer to 1, the better.
# We shall assume that in FMCG, tipically the elasticity is in the range [-5, 0]. 
# One could change this range to adapt it to other products, but it is not the purpose of this notebook.
def compute_elasticity_score(model, val_loader, device, 
                             own_min=-5.0, own_max=0.0,
                             cross_min=-1.0, cross_max=1.0):
    model.eval()
    all_own, all_cross = [], []
    off_diag = ~torch.eye(model.n, dtype=torch.bool, device=device).unsqueeze(0)  # (1, n, n)


    with torch.no_grad():
        for batch in val_loader:
            batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            obs_mask = batch["obs_mask"].bool() & batch["price_observed"].bool()
            _, eps_hat, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)

            # Elasticity matrix
            E = aux["E"]
            # Own-price elasticity
            all_own.append(eps_hat[obs_mask].cpu()) 
            # Cross-price elasticity
            # We get the pair mask (B, n, n)
            cross_mask = active_cross_mask(E, obs_mask, aux["pairs"])
            all_cross.append(E[cross_mask].cpu())

    own = torch.cat(all_own).numpy() # Concatenate the own-price elasticities
    cross = torch.cat(all_cross).numpy() if all_cross else np.array([])

    # ── Own score ────────────────────────────────────────────────
    # We compute the percentage of predicted elasticities that are in the range [-5, 0].
    own_in_range  = float(((own >= own_min) & (own <= own_max)).mean())
    median_own    = float(np.median(own))
    # We compute the penalty for the prior.
    # Because of EDA, the global elasticity is -2 approximately.
    # Therefore, we want the median of the predicted elasticities to be -2.
    # If it is not, we penalize the model.
    deviation     = max(0.0, abs(median_own - BETA_EDA) - 0.3)
    prior_penalty = min(deviation / abs(BETA_EDA), 1.0)
    # It is a weighted average of the percentage of predicted elasticities in the range [-5, 0]
    # and the penalty for the prior.
    own_score     = own_in_range * (1.0 - prior_penalty)

    # ── Cross score ───────────────────────────────────────────────
    if len(cross) > 0:
        cross_in_range = float(((cross >= cross_min) & (cross <= cross_max)).mean())
        median_cross    = float(np.median(cross))
    else:
        # If there are no cross-price elasticities, we assume the score is 1.0
        cross_in_range = 1.0
        media_cross = float("nan")  

    # ── Final score ───────────────────────────────────────────────
    score = 0.7 * own_score + 0.3 * cross_in_range

    return {
        "elast_score":            float(score),
        "own_score":              float(own_score),
        "own_in_range":           float(own_in_range),
        "own_elasticity_median":  median_own,
        "cross_in_range":         float(cross_in_range),
        "cross_elasticity_median": median_cross,
    }

print("Helpers of metrics defined")

Helpers of metrics defined


In [ ]:
# This function build the model and train it.
# We encapsulate the training loop in a function to be able to use it in Optuna.
def build_and_train(params, train_fold, val_fold, fold_id, seed, trial_id=0):
    set_all_seeds(seed) # Set the seeds for reproducibility

    # Build the dataframes:
    # train_wide_s, val_wide_s are the smoothed dataframes.
    # train_wide, val_wide are the original dataframes
    # The four dataframes have the store and week columns encoded.
    train_wide, val_wide, train_wide_s, val_wide_s = build_fold_frames(
        train_wide=train_fold,
        val_wide=val_fold,
        smooth_window=SMOOTH_WINDOW,
    )

    # Build DataSets (Pytorch) for the phase0, phase1 and phase2.
    #  · train_ds_p0, val_ds_p0 are the DataSets for the phase0.
    #  · train_ds, val_ds are the DataSets for the phase1 and phase2.
    # Important! The dataframes _s are the smoothed dataframes and are only used
    # to compute the train_ds_p0 and val_ds_p0. Therefore, for the phase0
    # our objective is to fit the model to the smoothed dataframes and get the
    # best parameters c(x) and beta(x) without the splines activated.
    train_loader_p0, val_loader_p0, train_loader, val_loader= build_loaders(
        train_wide, val_wide, train_wide_s, val_wide_s, batch_size=params["BATCH_SIZE"]
    )

    # ------ IMPORTANT------
    # We need to emphasize the following:
    # in the build_fold_frames function is the encoder of store_code done;
    # Remember that this encoding is continous, namely, it goes from [101, 205, 312]
    # to [0, 1, 2]. It is extremly important not to reorder this encoding, because
    # the following is thought/computed/coded assuming this order. For instance,
    # in the MultiProductContextEmbeddings, the store_code is used to index the
    # store embedding. If you reorder the encoding, you will be using the wrong
    # embedding for the store.
    # -----------------------

    # Get the parameters from the Optuna trial.
    n_basis          = params["N_BASIS"]
    hidden           = HIDDEN_OPTIONS[params["HIDDEN_KEY"]]
    dropout          = params["DROPOUT"]
    act              = params.get("ACT", "gelu")
    lr_p0            = params["LR_P0"]
    lr_p1            = params["LR_P1"]
    lambda_smooth = params["LAMBDA_SMOOTH"]
    lambda_elast  = params["LAMBDA_ELAST"]


    # Define the paths to the checkpoints for the phase0 and phase1.
    ckpt_p0 = CKPT_DIR / f"trial{trial_id}_fold{fold_id}_seed{seed}_phase0.pt"
    ckpt_p1 = CKPT_DIR / f"trial{trial_id}_fold{fold_id}_seed{seed}_phase1.pt"

    # ── BUILD THE MODEL ───────────────────────────────────────

    # Build the knots for the splines.
    builder = SplineBuilder()
    spline_configs = []
    for i in range(n_upcs):
        obs = train_wide[f"price_observed_{i}"].astype(bool)
        x_i = train_wide.loc[obs, f"log_price_{i}"].values
        if len(x_i) < 10:
            raise ValueError(f"Too few observed prices for UPC {i} to build splines")
        config = builder.build_from_data(
            x_i, n_basis=n_basis, q_min=0.05, q_max=0.95, basis_type="truncated_cubic")
        spline_configs.append(config)

    # Build the price splines (Theory implementation): Bx, dBx, ddBx.
    price_splines = build_price_basis("truncated_cubic", spline_configs)

    # Build the context embeddings for each product (token).
    # We get a (B, out_dim) context tensor. In the article, this tensor is called x_i.
    token_builder = ProductTokenBuilder(
        n=n_upcs,
        n_stores=n_stores, d_store=D_STORE,
        n_brands=n_brands, d_brand=D_BRAND,
        n_styles=n_styles, d_style=D_STYLE,
    )

    # Build the all-in-one model. All the pieces together.
    def make_model(enforce_negative_beta, use_cross):
        # From the latent representation h,
        # the model computes the parameters b, beta, w, u.
        # Finally, it computes the predicted demand y_hat,
        # the own-price elasticity eps_hat, and the elasticity matrix E.
        head = IntegrableDemandHead(
            context_dim=token_builder.d_token,
            K_splines=n_basis,
            n=n_upcs,
            k_neighbors=K_NEIGHBORS,
            hidden=hidden,
            act=act,
            dropout=dropout,
            use_cross=use_cross,
            enforce_negative_beta=enforce_negative_beta,
        )
        # The model is built. All the pieces together.
        return ICDN(
            context_builder=token_builder,
            price_splines=price_splines,
            head=head,
            n=n_upcs,
        ).to(device)

    # ── PHASE 0 ─────────────────────────────────────────────────────
    # The goal of this phase is to obtain a robust initialization before
    # unlocking the model's full flexibility. To do so:
    #
    #   1. First-order cross-price effects are able to be computed (use_cross=True)
    #      and the spline weights are frozen (head_w → zeros, requires_grad=False).
    #      This reduces the model to a log-linear demand:
    #       log(q) \approx b + beta·log(p) + first-order cross-price effects.
    #
    #   2. The head_beta bias is initialized with the inverse softplus of
    #      BETA_EDA, so that the own-price elasticity at startup equals exactly
    #      -BETA_EDA. This gives the model an economically sensible starting
    #      point instead of a random one.
    #
    #   3. The loss applies no smoothness or positivity penalties (lambda_smooth=0,
    #      lambda_pos=0): only the demand prediction error is minimized.
    #
    # By the end of this phase, beta and b are well calibrated, which makes
    # convergence easier in later phases when spline weights and cross-price
    # effects are unfrozen.

    # Build the model with first-order cross-price effects and enforcing negative beta.
    m0 = make_model(enforce_negative_beta=True, use_cross=True)
    # Zero and freeze spline / bilinear heads: Phase 0 is log-linear.
    zero_and_freeze_nonlinear(m0)
    # We initialize the head_beta bias with the inverse softplus of BETA_EDA.
    init_beta_prior(m0, BETA_EDA)
    with torch.no_grad():
        # Zero-init head_beta_cross and head_w_cross so that cross-price
        # contributions start at zero and are learned gradually from phase 1 onward.
        m0.head.param_head.head_beta_cross.weight.zero_()
        m0.head.param_head.head_beta_cross.bias.zero_()

    # Define the loss function for the phase0. Notice that we use the mean reduction and
    # only focus on the accuracy of the demand prediction (huber_delta != 0).
    loss_p0 = ElasticityLoss(
        huber_delta=1.0,
        lambda_smooth=lambda_smooth,
        lambda_elast=lambda_elast,
        reduction="mean"
    )

    # We define the optimizer for the phase0. We use AdamW with a weight decay of 1e-5.
    # For bias, we don't use weight decay, and for head_w and head_cross, neither,
    # since these weights are already regularized by lambda_smooth, therefore,
    # it would be double regularization.
    # Let us see it:
    # AdamW: L_total = L_task + \lambda · ||w||^2
    # Smooth: L_smooth = \lambda_smooth · mean ( (w · ddBx)^2 + ... )
    # Total: L_lotal = L_huber + L_positivity + \lambda_smooth · mean ( (w · ddBx)^2 + ... ) + \lambda · ||w||^2
    # We see then that the weight decay is applied twice, for smoothness and for the weights.
    decay, no_decay = [], []
    for name, p in m0.named_parameters():
        if not p.requires_grad:
            continue
        if (("head_w" in name) or ("head_cross" in name)
                or ("head_beta_cross" in name) or name.endswith("bias")):
            no_decay.append(p)
        else:
            decay.append(p)

    # Define AdamW phase0.
    opt_p0 = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-5},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=lr_p0,
    )

    # Define the scheduler for the phase0.
    # Mode = "min" means that the learning rate will be reduced when the validation loss
    # does not improve for PATIENCE epochs.
    # Factor = 0.5 means that the learning rate will be reduced by a factor of 0.5.
    # Patience = 10 means that the learning rate will be reduced after 10 epochs of no improvement.
    # Min_lr = 1e-5 means that the learning rate will not be reduced below 1e-5.
    sch_p0 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p0, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5
    )

    # Run the training for the phase0. Inner-val SÍ se usa aquí (selección, no reporte).
    hist0 = run_training(m0, train_loader_p0, val_loader_p0, loss_p0,
                     opt_p0, sch_p0, N_EPOCHS_P0, ES_PATIENCE,
                     ckpt_p0, device, neighbor_meta, "P0")

    # ── Phase 1: Unlock spline weights with smoothed targets ───────────────────
    # Building on the stable beta and b from Phase 0, this phase introduces the
    # spline flexibility that was previously frozen:
    #
    #   1. The model is initialized from the Phase 0 checkpoint. The spline
    #      weights (head_w) are unfrozen (requires_grad=True), allowing the
    #      model to learn non-linear price responses beyond the log-linear baseline.
    #
    #   2. Training uses the non-smoothed data (train_loader / val_loader),
    #      unlike Phase 0 which trained on rolling-average targets.
    #
    # By the end of this phase, the spline shapes are well fit to the raw demand
    # signal.

    # Build the model with first-order and second-order cross-price effects
    # and enforcing negative beta.
    m1 = make_model(enforce_negative_beta=True, use_cross=True)
    # Load the state dict from the Phase 0 checkpoint.
    m1.load_state_dict(torch.load(ckpt_p0, map_location=device))
    # Unfreeze the nonlinear parameters.
    unfreeze_nonlinear(m1)

    # We define the loss function for the phase1. Pure fit to the training data.
    loss_p1 = ElasticityLoss(
        huber_delta=1.0,
        lambda_smooth=lambda_smooth,
        lambda_elast=lambda_elast,
        reduction="mean"
    )
    # The same as before. Avoiding double regularization.
    decay, no_decay = [], []
    for name, p in m1.named_parameters():
        if not p.requires_grad:
            continue
        if (("head_w" in name) or ("head_cross" in name)
                or ("head_beta_cross" in name) or name.endswith("bias")):
            no_decay.append(p)
        else:
            decay.append(p)

    # Define AdamW phase1. As before.
    opt_p1 = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-5},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=lr_p1,
    )

    # Define the scheduler for the phase1. As before.
    sch_p1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p1, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5
    )

    # Run the training for the phase1. Inner-val SÍ se usa aquí.
    hist1 = run_training(m1, train_loader, val_loader, loss_p1,
                     opt_p1, sch_p1, N_EPOCHS_P1, ES_PATIENCE,
                     ckpt_p1, device, neighbor_meta, "P1")

    # Load the state dict from the Phase 1 checkpoint.
    m1.load_state_dict(torch.load(ckpt_p1, map_location=device))

    # Freeze P* once on the converged model: compute the global mean score matrix
    # over the full training set, then fix the sparse neighbor graph.
    # From this point on, run() uses the O(B * E * d_attn) sparse path.
    m1.eval()
    selector = m1.head.neighbor_selector
    def h_iter(loader):
        with torch.no_grad():
            for batch in loader:
                batch  = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
                tokens = m1.context_builder(batch)   # (B, n, d_token)
                h      = m1.head.encoder(tokens)     # (B, n, d_hidden)
                yield h
    global_mean = selector.accumulate_mean_scores(
        h_iter(train_loader),
        category=neighbor_meta["category"],
        brand=neighbor_meta["brand"],
        style=neighbor_meta["style"],
        liters=neighbor_meta["liters"],
    )
    selector.freeze_graph(
        global_mean,
        category=neighbor_meta["category"],
        brand=neighbor_meta["brand"],
        style=neighbor_meta["style"],
        liters=neighbor_meta["liters"],
    )

    # Compute the prediction metrics for the global metrics and the elasticity score.
    pred_metrics = compute_global_metrics(m1, val_loader, device)
    elast_metrics = compute_elasticity_score(m1, val_loader, device)

    out = {
        "trial_id": trial_id,
        "fold": fold_id,
        "seed": seed,
        "n_train": len(train_wide),
        "n_val": len(val_wide),
        "best_epoch_p0": hist0["best_epoch"],
        "best_epoch_p1": hist1["best_epoch"],
        **pred_metrics,
        **elast_metrics,
    }

    print(
        f"trial={trial_id} fold={fold_id} seed={seed} | "
        f"R2={out['r2_val']:.4f} MAE={out['mae_val']:.4f} | "
        f"ElastScore={out['elast_score']:.4f} | "
        f"ep0={out['best_epoch_p0']} ep1={out['best_epoch_p1']} | "
        f"own[pct={100*out['own_in_range']:.1f}% med={out['own_elasticity_median']:.2f}] "
        f"cross[pct={100*out['cross_in_range']:.1f}% med={out['cross_elasticity_median']:.2f}]"
    )
    # Remove the checkpoint files. For each trial, we have 2 checkpoints.
    # If we don't remove them, the folder will be full of checkpoints and our
    # hard drive will run out of space.
    ckpt_p0.unlink(missing_ok=True)
    ckpt_p1.unlink(missing_ok=True)

    return out

print("build_and_train redefinided")

# Optuna Study

In [ ]:
# Objective function to optimize in the hyperparameter search (Optuna).
trial_records = []

def objective(trial, fold_splits, study_tag: str):
    # Define the parameters to optimize.
    params = {
        "N_BASIS":             trial.suggest_int("N_BASIS", 2, 16),
        "HIDDEN_KEY":          trial.suggest_categorical("HIDDEN_KEY", list(HIDDEN_OPTIONS.keys())),
        "DROPOUT":             trial.suggest_float("DROPOUT", 0.0, 0.3),
        "LR_P0":               trial.suggest_float("LR_P0", 1e-4, 1e-2, log=True),
        "LR_P1":               trial.suggest_float("LR_P1", 1e-5, 5e-3, log=True),
        "LAMBDA_SMOOTH": trial.suggest_float("LAMBDA_SMOOTH", 1e-5, 0.2, log=True),
        "LAMBDA_ELAST":  trial.suggest_float("LAMBDA_ELAST",  1e-5, 0.2, log=True),
        "BATCH_SIZE":          trial.suggest_categorical("BATCH_SIZE", [256, 512, 1024]),
    }

    print(f"\n{'='*70}")
    print(f"[{study_tag}] Trial {trial.number}")
    for k, v in params.items():
        print(f"  {k}: {v}")
    print(f"{'='*70}")

    # Run the training for each INNER fold and seed.
    run_rows = []
    for fold_id, (train_fold, val_fold) in enumerate(fold_splits):
        for seed in TUNE_SEEDS:
            row = build_and_train(
                params=params,
                train_fold=train_fold,
                val_fold=val_fold,
                fold_id=fold_id,
                seed=seed,
                trial_id=trial.number,
            )
            run_rows.append(row)

    # Create a DataFrame from the run_rows.
    df_trial = pd.DataFrame(run_rows)

    # Compute the mean and standard deviation of the R2.
    mean_r2 = float(df_trial["r2_val"].mean())
    std_r2  = float(df_trial["r2_val"].std(ddof=1)) if len(df_trial) > 1 else 0.0

    # Compute the mean and standard deviation of the Elasticity Score.
    mean_elast = float(df_trial["elast_score"].mean())
    std_elast  = float(df_trial["elast_score"].std(ddof=1)) if len(df_trial) > 1 else 0.0

    # Compute the mean and standard deviation of the MAE.
    mean_mae  = float(df_trial["mae_val"].mean())
    mean_rmse = float(df_trial["rmse_val"].mean())

    # Compute the robust score. We try to penalize the variance between folds
    # and rewards those trials that are more stable across folds. We set 0.25
    # to control how much we penalize the variance.
    robust_r2 = mean_r2 - 0.25 * std_r2
    robust_elast = mean_elast - 0.25 * std_elast

    # Set the user attributes for the trial.
    trial.set_user_attr("mean_r2", mean_r2)
    trial.set_user_attr("std_r2", std_r2)
    trial.set_user_attr("mean_elast_score", mean_elast)
    trial.set_user_attr("std_elast_score", std_elast)
    trial.set_user_attr("mean_mae", mean_mae)
    trial.set_user_attr("mean_rmse", mean_rmse)
    trial.set_user_attr("robust_r2", robust_r2)
    trial.set_user_attr("robust_elast", robust_elast)

    # We build the historical records for the trials because, at the end,
    # we want to analyze the performance of the trials.
    df_trial["trial"] = trial.number
    df_trial["study_tag"] = study_tag
    for k, v in params.items():
        df_trial[k] = v
    trial_records.extend(df_trial.to_dict(orient="records"))

    print(
        f"[{study_tag}] Trial {trial.number} summary | "
        f"mean_R2={mean_r2:.4f} std_R2={std_r2:.4f} "
        f"robust_R2={robust_r2:.4f} | "
        f"mean_Elast_Score={mean_elast:.4f} std_Elast_Score={std_elast:.4f} "
        f"robust_Elast_Score={robust_elast:.4f}"
    )

    return robust_r2, robust_elast


def study_to_summary(study) -> pd.DataFrame:
    summary_rows = []
    for t in study.trials:
        if t.values is None:
            continue
        summary_rows.append({
            "trial": t.number,
            "mean_r2": t.user_attrs.get("mean_r2", np.nan),
            "std_r2": t.user_attrs.get("std_r2", np.nan),
            "robust_r2": t.user_attrs.get("robust_r2", np.nan),
            "robust_elast": t.user_attrs.get("robust_elast", np.nan),
            "mean_elast_score": t.user_attrs.get("mean_elast_score", np.nan),
            "std_elast_score": t.user_attrs.get("std_elast_score", np.nan),
            "mean_mae": t.user_attrs.get("mean_mae", np.nan),
            "mean_rmse": t.user_attrs.get("mean_rmse", np.nan),
            **t.params,
        })
    df = pd.DataFrame(summary_rows)
    df["robust_score"] = df["robust_r2"].fillna(0.0) + df["robust_elast"].fillna(0.0)
    return df.sort_values("robust_score", ascending=False)


def best_payload_icdn(best_row: pd.Series) -> dict:
    return {
        "trial": int(best_row["trial"]),
        "robust_score": float(best_row["robust_score"]),
        "mean_r2": float(best_row["mean_r2"]),
        "std_r2": float(best_row["std_r2"]),
        "mean_elast_score": float(best_row["mean_elast_score"]),
        "std_elast_score": float(best_row["std_elast_score"]),
        "params": {
            "N_BASIS":       int(best_row["N_BASIS"]),
            "HIDDEN_KEY":    str(best_row["HIDDEN_KEY"]),
            "DROPOUT":       float(best_row["DROPOUT"]),
            "LR_P0":         float(best_row["LR_P0"]),
            "LR_P1":         float(best_row["LR_P1"]),
            "LAMBDA_SMOOTH": float(best_row["LAMBDA_SMOOTH"]),
            "LAMBDA_ELAST":  float(best_row["LAMBDA_ELAST"]),
            "BATCH_SIZE":    int(best_row["BATCH_SIZE"]),
        },
    }


def median_best_epochs(records, trial: int, study_tag: str) -> dict:
    rec = pd.DataFrame(records)
    sub = rec[(rec["trial"] == trial) & (rec["study_tag"] == study_tag)]
    if sub.empty or "best_epoch_p0" not in sub.columns:
        raise ValueError(f"No inner epochs for trial={trial} tag={study_tag}")
    return {
        "best_epoch_p0": int(np.round(sub["best_epoch_p0"].median())),
        "best_epoch_p1": int(np.round(sub["best_epoch_p1"].median())),
    }


def run_optuna(fold_splits, tag: str, n_trials: int):
    study = optuna.create_study(
        directions=["maximize", "maximize"],
        study_name=tag,
        storage=f"sqlite:///{NESTED_DIR / (tag + '.db')}",
        load_if_exists=True,
    )
    n_done = len([t for t in study.trials if t.values is not None])
    n_left = max(0, n_trials - n_done)
    print(f"{tag}: {n_done} done, {n_left} left")
    if n_left:
        # BE CAREFUL: 1 trial can take ~15 min on RTX 5070 Ti
        study.optimize(
            lambda t, splits=fold_splits, tag=tag: objective(t, splits, tag),
            n_trials=n_left,
        )
    return study

In [ ]:
outer_best = []

for plan in nested_plans:
    k = plan["outer_id"]
    tag = f"icdn_nested_outer{k}"
    study = run_optuna(plan["inner_splits"], tag, N_TRIALS_NESTED)
    summary = study_to_summary(study)
    summary["outer_id"] = k
    payload = best_payload_icdn(summary.iloc[0])
    payload["outer_id"] = k
    payload["protocol"] = PROTOCOL
    payload.update(median_best_epochs(trial_records, payload["trial"], tag))
    with open(NESTED_DIR / f"icdn_outer{k}_best_params.json", "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    summary.to_csv(NESTED_DIR / f"icdn_outer{k}_trials.csv", index=False)
    outer_best.append(payload)
    print(
        f"\nSaved {tag} best trial {payload['trial']} "
        f"robust_score={payload['robust_score']:.4f} "
        f"ep0={payload['best_epoch_p0']} ep1={payload['best_epoch_p1']}"
    )

with open(NESTED_DIR / "icdn_outer_best_params.json", "w", encoding="utf-8") as f:
    json.dump(outer_best, f, indent=2, ensure_ascii=False)

print("\nAll outer studies completed")

# Summary

In [ ]:
df_outer_best = pd.DataFrame([
    {
        "outer_id": p["outer_id"],
        "trial": p["trial"],
        "robust_score": p["robust_score"],
        "mean_r2": p["mean_r2"],
        "std_r2": p["std_r2"],
        "mean_elast_score": p["mean_elast_score"],
        "std_elast_score": p["std_elast_score"],
        **p["params"],
    }
    for p in outer_best
])
print(df_outer_best.to_string(index=False))

# Best Trial

In [ ]:
tag = "icdn_nested_holdout"
study_h = run_optuna(holdout_inner, tag, N_TRIALS_HOLDOUT)
holdout_summary = study_to_summary(study_h)
holdout_payload = best_payload_icdn(holdout_summary.iloc[0])
holdout_payload["protocol"] = PROTOCOL
holdout_payload["split"] = "holdout_80_20"
holdout_payload.update(median_best_epochs(trial_records, holdout_payload["trial"], tag))

with open(NESTED_DIR / "icdn_holdout_best_params.json", "w", encoding="utf-8") as f:
    json.dump(holdout_payload, f, indent=2, ensure_ascii=False)
holdout_summary.to_csv(NESTED_DIR / "icdn_holdout_trials.csv", index=False)

print("Holdout best saved in:", NESTED_DIR / "icdn_holdout_best_params.json")
print(json.dumps(holdout_payload, indent=2, ensure_ascii=False))